# Week 6: Visualization of Model Performance Metrics & Interpretation
## AI-Based Chocolate Paan Sales Prediction System
**Target:** `Boxes_Sold`  
**Dataset:** `Chocolate_Paan_Sales_Dataset_10000.xlsx` (10,000 samples)  
**Objective:** Visualizing Week 5 model benchmarking metrics, generalization analysis, actual vs predicted scatter, residual diagnostics, and feature interpretations.

In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Styling configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 150

print('Week 6 Visualization Environment Initialized Successfully.')

### 1. Load Actual Week 5 Model Evaluation Metrics
Retrieving the exact metrics computed in Week 5 benchmark from `model/metrics.json`.
**Models evaluated:**
1. Linear Regression
2. Decision Tree Regressor
3. Random Forest Regressor
4. Support Vector Regressor (RBF Kernel)
5. Gradient Boosting Regressor

In [ ]:
with open('model/metrics.json', 'r') as f:
    metrics_meta = json.load(f)

metrics_df = pd.DataFrame(metrics_meta['metrics']).sort_values(by='R2', ascending=False)
gen_df = pd.DataFrame(metrics_meta['generalization']).set_index('Model').loc[metrics_df['Model']].reset_index()

print('=== WEEK 5 MODEL PERFORMANCE BENCHMARK (TEST SET - 2,000 SAMPLES) ===')
display(metrics_df)
print('\n=== GENERALIZATION & OVERFITTING ANALYSIS ===')
display(gen_df)

### 2. Model Performance Comparison Table

| Model | MAE (Boxes) | MSE | RMSE (Boxes) | R² Score |
| :--- | :---: | :---: | :---: | :---: |
| **Linear Regression** | **7.7102** | **79.7161** | **8.9284** | **0.9449** |
| **Gradient Boosting** | 9.0932 | 121.3074 | 11.0140 | 0.9161 |
| **SVR (RBF Kernel)** | 9.3598 | 144.5780 | 12.0241 | 0.9001 |
| **Random Forest** | 11.7753 | 216.5535 | 14.7158 | 0.8503 |
| **Decision Tree** | 18.7780 | 553.4700 | 23.5259 | 0.6174 |

> **Key Takeaway:** Linear Regression is the best performing model with the highest R² (0.9449) and lowest MAE (7.7102 boxes) and RMSE (8.9284 boxes).

### 3. R² Comparison Graph
Comparing the coefficient of determination (R²) across all 5 models. Higher is better.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bar_colors = ['#2E7D32', '#1565C0', '#D99A3D', '#6A1B9A', '#C62828']
bars = ax.bar(metrics_df['Model'], metrics_df['R2'], color=bar_colors, width=0.55, edgecolor='#3B1F1F')
ax.set_title('Model Comparison — R² Score (Higher is Better)', fontsize=14, fontweight='bold', color='#3B1F1F', pad=15)
ax.set_ylabel('R² Score', fontsize=11, fontweight='bold', color='#3B1F1F')
ax.set_ylim(0, 1.08)
for bar in bars:
    h = bar.get_height()
    ax.annotate(f'{h:.4f}', xy=(bar.get_x() + bar.get_width()/2, h), xytext=(0, 4),
                textcoords='offset points', ha='center', va='bottom', fontweight='bold', color='#3B1F1F')
plt.xticks(rotation=15, ha='right', fontweight='bold')
plt.tight_layout()
plt.show()

### 4. MAE and RMSE Comparison Graphs
Lower error indicates closer predictions to actual sales volume.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# MAE Comparison
df_mae = metrics_df.sort_values(by='MAE')
bars1 = ax1.bar(df_mae['Model'], df_mae['MAE'], color=bar_colors, width=0.55, edgecolor='#3B1F1F')
ax1.set_title('Mean Absolute Error (MAE) — Lower is Better', fontsize=12, fontweight='bold', color='#3B1F1F')
ax1.set_ylabel('MAE (Boxes Sold)', fontweight='bold')
ax1.set_ylim(0, max(df_mae['MAE']) * 1.18)
for b in bars1:
    h = b.get_height()
    ax1.annotate(f'{h:.2f}', xy=(b.get_x() + b.get_width()/2, h), xytext=(0, 3),
                 textcoords='offset points', ha='center', va='bottom', fontweight='bold')
ax1.set_xticks(range(len(df_mae)))
ax1.set_xticklabels(df_mae['Model'], rotation=15, ha='right', fontweight='bold')

# RMSE Comparison
df_rmse = metrics_df.sort_values(by='RMSE')
bars2 = ax2.bar(df_rmse['Model'], df_rmse['RMSE'], color=bar_colors, width=0.55, edgecolor='#3B1F1F')
ax2.set_title('Root Mean Squared Error (RMSE) — Lower is Better', fontsize=12, fontweight='bold', color='#3B1F1F')
ax2.set_ylabel('RMSE (Boxes Sold)', fontweight='bold')
ax2.set_ylim(0, max(df_rmse['RMSE']) * 1.18)
for b in bars2:
    h = b.get_height()
    ax2.annotate(f'{h:.2f}', xy=(b.get_x() + b.get_width()/2, h), xytext=(0, 3),
                 textcoords='offset points', ha='center', va='bottom', fontweight='bold')
ax2.set_xticks(range(len(df_rmse)))
ax2.set_xticklabels(df_rmse['Model'], rotation=15, ha='right', fontweight='bold')

plt.tight_layout()
plt.show()

### 5. Training vs Testing R² (Generalization & Overfitting Diagnosis)
Comparing fit between training set (8,000 rows) and testing set (2,000 rows).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(gen_df['Model']))
width = 0.35

r1 = ax.bar(x - width/2, gen_df['Training R2'], width, label='Training R²', color='#5A2D22', edgecolor='#3B1F1F')
r2 = ax.bar(x + width/2, gen_df['Testing R2'], width, label='Testing R²', color='#D99A3D', edgecolor='#3B1F1F')

ax.set_title('Training vs Testing R² — Generalization & Overfitting Diagnosis', fontsize=14, fontweight='bold', color='#3B1F1F', pad=15)
ax.set_ylabel('R² Score', fontsize=11, fontweight='bold', color='#3B1F1F')
ax.set_xticks(x)
ax.set_xticklabels(gen_df['Model'], rotation=15, ha='right', fontweight='bold')
ax.set_ylim(0, 1.18)
ax.legend(loc='upper right', frameon=True)

for r in r1:
    h = r.get_height()
    ax.annotate(f'{h:.2f}', xy=(r.get_x() + r.get_width()/2, h), xytext=(0, 3),
                textcoords='offset points', ha='center', va='bottom', fontsize=9, fontweight='bold')
for r in r2:
    h = r.get_height()
    ax.annotate(f'{h:.2f}', xy=(r.get_x() + r.get_width()/2, h), xytext=(0, 3),
                textcoords='offset points', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

### 6. Actual vs Predicted and Residual Plot for Selected Model (Linear Regression)
Evaluating the calibration and residual errors of the production model.

In [ ]:
df = pd.read_excel('Chocolate_Paan_Sales_Dataset_10000.xlsx')
features = metrics_meta['features']
target = metrics_meta['target']

X_train, X_test, Y_train, Y_test = train_test_split(df[features], df[target], test_size=0.20, random_state=42)
lr_model = joblib.load('model/trained_model.pkl')
y_pred_test = lr_model.predict(X_test)
residuals = Y_test.values - y_pred_test

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Actual vs Predicted
np.random.seed(42)
idx = np.random.choice(len(Y_test), size=400, replace=False)
y_s = Y_test.values[idx]
p_s = y_pred_test[idx]
r_s = residuals[idx]

ax1.scatter(y_s, p_s, alpha=0.55, color='#D99A3D', edgecolors='#5A2D22', s=35, label='Test Samples')
min_v = min(y_s.min(), p_s.min()) - 10
max_v = max(y_s.max(), p_s.max()) + 10
ax1.plot([min_v, max_v], [min_v, max_v], color='#C62828', linestyle='--', linewidth=2, label='Ideal Reference (y = x)')
ax1.set_title('Actual vs Predicted Boxes Sold (Linear Regression, R² = 0.9449)', fontsize=12, fontweight='bold', color='#3B1F1F')
ax1.set_xlabel('Actual Boxes Sold', fontweight='bold')
ax1.set_ylabel('Predicted Boxes Sold', fontweight='bold')
ax1.legend()

# Residual Plot
ax2.scatter(p_s, r_s, alpha=0.55, color='#5A2D22', edgecolors='#3B1F1F', s=35)
ax2.axhline(0, color='#C62828', linestyle='--', linewidth=2, label='Zero Residual Baseline (y = 0)')
ax2.set_title('Residual Plot (Predicted vs Residuals)', fontsize=12, fontweight='bold', color='#3B1F1F')
ax2.set_xlabel('Predicted Boxes Sold', fontweight='bold')
ax2.set_ylabel('Residuals (Actual - Predicted)', fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

### 7. Feature Interpretation: Linear Regression Coefficients vs Random Forest Feature Importance

> **Methodological Difference:**
> - **Linear Regression Coefficients** have units and sign (+/-). A positive coefficient means an increase in the feature directly increases `Boxes_Sold`.
> - **Random Forest Feature Importance** represents the percentage contribution to impurity reduction (MDI) across all decision trees. It sums to 100% and does not denote directionality.
> They are presented in separate plots below as required by the SOP.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Linear Regression Coefficients
coef_series = pd.Series(metrics_meta['coefficients']).sort_values()
c_colors = ['#C62828' if v < 0 else '#2E7D32' for v in coef_series.values]
bars_c = ax1.barh(coef_series.index, coef_series.values, color=c_colors, edgecolor='#3B1F1F', height=0.6)
ax1.axvline(0, color='#3B1F1F', linewidth=1)
ax1.set_title(f'Linear Regression Coefficients (Intercept = {metrics_meta["intercept"]:.2f})', fontsize=12, fontweight='bold', color='#3B1F1F')
ax1.set_xlabel('Coefficient Value (Change in Boxes Sold)', fontweight='bold')
for bar in bars_c:
    w = bar.get_width()
    ha = 'left' if w >= 0 else 'right'
    ax1.annotate(f'{w:+.2f}', xy=(w, bar.get_y() + bar.get_height()/2),
                 xytext=(5 if w >= 0 else -5, 0), textcoords='offset points',
                 ha=ha, va='center', fontsize=9, fontweight='bold')

# Random Forest Feature Importance
rf_series = (pd.Series(metrics_meta['rf_importances']) * 100).sort_values()
bars_rf = ax2.barh(rf_series.index, rf_series.values, color='#D99A3D', edgecolor='#5A2D22', height=0.6)
ax2.set_title('Random Forest Feature Importance (% Relative Impurity Contribution)', fontsize=12, fontweight='bold', color='#3B1F1F')
ax2.set_xlabel('Relative Importance (%)', fontweight='bold')
for bar in bars_rf:
    w = bar.get_width()
    ax2.annotate(f'{w:.2f}%', xy=(w, bar.get_y() + bar.get_height()/2),
                 xytext=(5, 0), textcoords='offset points',
                 ha='left', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()